# Ingest results .json file
- 1.- Read the file using spark dataframe reader API 
- 2.- Define and enforce schem (preserve the nested structure)
- 3.- Add Metadata Columns
-     Source File
-     Ingestion Timestamp
- 4.- Write to bronze delta table 

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/02.bronze-helpers  

In [0]:
# Define source_file and table_name
source_file = f"{landing_folder_path}/results"
table_name = f"{catalog_name}.{bronze_schema}.results"

### 1.- Read the file using spark dataframe reader API

In [0]:
# Define the Schema
from pyspark.sql.types import StructType, StructField, StringType, DateType, IntegerType, FloatType

results_schema = StructType([
    StructField("date", DateType()),
    StructField("raceName", StringType()),
    StructField("round", IntegerType()),
    StructField("season", IntegerType()),
    StructField("url", StringType()),
    StructField("constructorId", StringType()),
    StructField("driverId", StringType()),
    StructField("grid", IntegerType()),
    StructField("laps", IntegerType()),
    StructField("number", IntegerType()),
    StructField("points", FloatType()),
    StructField("position", IntegerType()),
    StructField("positionText", StringType()),
    StructField("status", StringType())
    
])                 

### 2.- Define and enforce schem (preserve the nested structure)


In [0]:
# Read data from the results file
results_df = (
    spark.read
    .format('json')
    .schema(results_schema)
    .option('mode', 'FAILFAST')
    .load(source_file)
)

### 3.- Add Metadata Columns

In [0]:
results_final_df = add_ingestion_metadata(results_df)

### 4.- Write to bronze delta table

In [0]:
(
    results_final_df
    .write
    .format('delta')
    .mode('overwrite')
    .saveAsTable(table_name)
)

In [0]:
display(spark.read.table(table_name))

In [0]:
%sql
SELECT season, COUNT(*)
FROM formula1.bronze.results
GROUP BY season
ORDER BY season;